In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load tokenizer and model
model_name = "openai-community/gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
gpt2 = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [2]:
from transformers import pipeline
import torch
pipe = pipeline("text-generation", model="openai-community/gpt2")
prompt ="what is transformer in a LLM  "
output=pipe(prompt)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [3]:
print(output[0]["generated_text"])

what is transformer in a LLM   function? The answer is that there is a transformer in each function, which is called a transformer and it has the following properties. First, it has a power of 0.2  (1/2) which is the power that your transformer produces. Second, it has the following properties: It is a transformer of 2.5  (4.7) which is the power that your current transformer produces. Third, it has the following properties: It is a transformer of 4.5  (5.6) which is the power that your current transformer produces. Fourth, it has the following properties: It is the power of 1  (4.7) which is the power that your current transformer produces. Fifth, it has the following properties: It is the power of 0.2  (1/2) which is the power that your current transformer produces. Sixth, it has the following properties: It is the power of 1.5  (1/2) which is the power that your current transformer produces. Seventh, it has the following properties: It is the power of 6.5  (1/2) which is the power t

In [4]:
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained("gpt2")

In [5]:
sentence="i learn machine learning to enhance workplace"
token_ids=tokenizer(sentence,return_tensors="pt").input_ids
outputs=gpt2(token_ids).logits[0,-1]
final_logits=torch.topk(outputs,10)
final_logits
for index in final_logits.indices:
  print(tokenizer.decode(index))

 productivity
 safety
 performance
 and
 communication
 learning
 interactions
 security
 skills
 experience


In [6]:
# greedy decode sampling strategy
def greedy_decode(logits):
    """return token index with the highest probability"""
    return torch.argmax(logits,dim=-1)

tokenizer.decode(greedy_decode(outputs))

def top_k_sampling(logits,k=50):
    """keeps only top k logits, normalize them into probabilitiy.
    then sample one token form the filtered distribution.
    """
    values,indices=torch.topk(logits,k)
    probs=F.softmax(values,dim=-1)
    sampled=torch.multinomial(probs,1)
    return indices[sampled]

def top_p_sampling(logits,p=0.9):
    """
    sort tokens by probability, keep smallest number whoese culumative probability exceed p, then sample one token .
    """
    sorted_logits,sorted_indices=torch.sort(logits,descending=True)
    sorted_probs=F.softmax(sorted_logits,dim=-1)
    cumulative_probs=sorted_probs.cumsum(dim=-1)
    
    # mask token outside nuclues
    mask=cumulative_probs>p
    sorted_logits[mask]=float("-inf")
    
    # sample from the filtered logits
    filtered_probs=F.softmax(sorted_logits,dim=-1)
    sampled=torch.multinomial(filtered_probs,1)
    
    # return the token index in original vocabulary
    return sorted_indices[sampled]

    

In [7]:
top_k_sampling(outputs)
tokenizer.decode(top_k_sampling(outputs))

' efficiency'

In [8]:
top_p_sampling(outputs,p=0.9)
tokenizer.decode(top_p_sampling(outputs,p=0.9))

' and'

In [9]:
# temperature sampling

def temperature_sampling(logits,temperature=.5):
    """scale logits by temperature before sampling.
    lower temperature -> sharper distribution -> more deterministic approach."""
    scaled=logits/temperature
    probs=F.softmax(scaled,dim=-1)
    return torch.multinomial(probs,1)

In [10]:
tokenizer.decode(temperature_sampling(outputs,temperature=1))

' lives'

In [11]:
# randlom smapling

def random_sampling(logits):
    """
    sample direcely from softmax distribution without filtering or scaling.`
    """
    probs=F.softmax(logits,dim=-1)
    return torch.multinomial(probs,1)

random_sampling(outputs)
tokenizer.decode(random_sampling(outputs))


' productivity'

In [12]:
# all sampling 
sentence= "today i decided to go to the college and play vollyball with my"
inputs=tokenizer(sentence,return_tensors="pt")
output=gpt2(**inputs)
logits=output.logits[0,-1]

print("Greedy decode:",tokenizer.decode([greedy_decode(logits)])) 
print("Top-k decode:",tokenizer.decode(top_k_sampling(logits,k=10)))
print("Top-p decode:",tokenizer.decode(top_p_sampling(logits,p=0.9)))
print("Temperature decode:",tokenizer.decode(temperature_sampling(logits,temperature=0.7)))
print("Random decode:",tokenizer.decode(random_sampling(logits)))

Greedy decode:  friends
Top-k decode:  friends
Top-p decode:  mom
Temperature decode:  dad
Random decode:  friends


In [21]:
sentence="i learn machine learning to enhance our understanding"
token_id=tokenizer(sentence,return_tensors="pt").input_ids
outputs=gpt2(token_id).logits
outputs=torch.softmax(outputs[0,-1],dim=-1)

top10=torch.topk(outputs,10)
for index, value in zip(top10.indices, top10.values):
    print(f"{tokenizer.decode(index)} -- {value:.1%}")

 of -- 90.6%
 and -- 3.5%
. -- 1.4%
, -- 1.0%
 about -- 0.6%
 in -- 0.3%
 for -- 0.2%
 on -- 0.2%
 by -- 0.2%
 as -- 0.1%
